[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/paper_implementation/blob/main/mean_flow_sanity_check.ipynb)

# Conditional MeanFlow — MNIST + DiT + Muon

기존 MeanFlow/JVP 식은 유지하고, MNIST class condition을 추가했다. DiT attention/MLP의 2D hidden matrix는 Muon, 나머지 parameter는 AdamW로 학습한다. `norm_eps=0.01`. 생성 시 `CONDITION_POOL`에서 label을 매번 랜덤 선택하고 PNG 위에 target label을 표시한다.


## 0. Setup / data


In [ ]:
!pip -q install datasets tensorboard
import math, os, random, time, warnings
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from torch.func import jvp
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from torchvision import datasets as tv_datasets, transforms
from torchvision.transforms import ToTensor

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if not torch.cuda.is_available(): raise RuntimeError("Use a Colab GPU runtime.")
DEVICE=torch.device("cuda"); GPU_NAME=torch.cuda.get_device_name(0)
print("GPU:", GPU_NAME)
torch.backends.cuda.matmul.allow_tf32=True
torch.backends.mha.set_fastpath_enabled(False)
TRAIN_STEPS=20_000; BATCH_SIZE=128; DIAG_BATCH=64
MUON_LR=2e-2; ADAMW_LR=3e-4; MUON_MOMENTUM=0.95
P_MEAN=-0.4; P_STD=1.0; DATA_PROPORTION=0.75; NORM_EPS=0.01
NUM_CLASSES=10; CONDITION_POOL=tuple(range(NUM_CLASSES))
SAMPLE_EVERY=1000; DIAG_EVERY=250; LOG_EVERY=50
ROOT="/content/meanflow_mnist_dit_sanity"
RUN="mnist_cond_muon20k_"+time.strftime("%Y%m%d_%H%M%S")
RUN_DIR=os.path.join(ROOT,RUN); SAMPLE_DIR=os.path.join(RUN_DIR,"samples"); CKPT_DIR=os.path.join(RUN_DIR,"checkpoints")
LOG_DIR=os.path.join(ROOT,"tensorboard",RUN)
for p in (SAMPLE_DIR,CKPT_DIR,LOG_DIR): os.makedirs(p,exist_ok=True)
writer=SummaryWriter(LOG_DIR)

to_tensor=ToTensor()
def transform_image(pil):
    x=to_tensor(pil); x=F.pad(x,(2,2,2,2),value=0.0); return x*2-1

def collate(batch):
    return torch.stack([transform_image(b["image"]) for b in batch]), torch.tensor([b["label"] for b in batch],dtype=torch.long)

try:
    ds=load_dataset("ylecun/mnist")
    train_loader=DataLoader(ds["train"],batch_size=BATCH_SIZE,shuffle=True,num_workers=2,pin_memory=True,drop_last=True,collate_fn=collate)
    test_loader=DataLoader(ds["test"],batch_size=DIAG_BATCH,shuffle=False,num_workers=2,pin_memory=True,collate_fn=collate)
    print("MNIST: Hugging Face")
except Exception as e:
    print("HF failed, torchvision fallback:",repr(e))
    tfm=transforms.Compose([transforms.ToTensor(),transforms.Pad(2),transforms.Normalize((0.5,),(0.5,))])
    tr=tv_datasets.MNIST("/content/mnist_data",train=True,transform=tfm,download=True)
    te=tv_datasets.MNIST("/content/mnist_data",train=False,transform=tfm,download=True)
    train_loader=DataLoader(tr,batch_size=BATCH_SIZE,shuffle=True,num_workers=2,pin_memory=True,drop_last=True)
    test_loader=DataLoader(te,batch_size=DIAG_BATCH,shuffle=False,num_workers=2,pin_memory=True)


## 1. Conditional DiT


In [ ]:
class ScalarEmbed(nn.Module):
    def __init__(self,d,f=128):
        super().__init__(); self.f=f; self.mlp=nn.Sequential(nn.Linear(f,d),nn.SiLU(),nn.Linear(d,d))
    def forward(self,s):
        h=self.f//2; freq=torch.exp(-math.log(10000)*torch.arange(h,device=s.device,dtype=s.dtype)/h)
        a=s[:,None]*freq[None,:]*2*math.pi
        return self.mlp(torch.cat([a.cos(),a.sin()],-1))

class Block(nn.Module):
    def __init__(self,d=224,heads=8):
        super().__init__(); self.n1=nn.LayerNorm(d,elementwise_affine=False,eps=1e-6); self.n2=nn.LayerNorm(d,elementwise_affine=False,eps=1e-6)
        self.attn=nn.MultiheadAttention(d,heads,batch_first=True)
        self.mlp=nn.Sequential(nn.Linear(d,4*d),nn.GELU(approximate="tanh"),nn.Linear(4*d,d))
        self.mod=nn.Sequential(nn.SiLU(),nn.Linear(d,6*d)); nn.init.zeros_(self.mod[-1].weight); nn.init.zeros_(self.mod[-1].bias)
    def forward(self,x,c):
        s1,k1,g1,s2,k2,g2=self.mod(c).chunk(6,-1)
        q=self.n1(x)*(1+k1[:,None,:])+s1[:,None,:]
        a,_=self.attn(q,q,q,need_weights=True); x=x+g1[:,None,:]*a
        q=self.n2(x)*(1+k2[:,None,:])+s2[:,None,:]
        return x+g2[:,None,:]*self.mlp(q)

class CondDiT(nn.Module):
    def __init__(self,d=224,depth=4,patch=4):
        super().__init__(); self.patch=patch; self.pe=nn.Conv2d(1,d,patch,patch); self.pos=nn.Parameter(torch.zeros(1,64,d))
        self.te=ScalarEmbed(d); self.he=ScalarEmbed(d); self.ye=nn.Embedding(10,d)
        self.blocks=nn.ModuleList([Block(d,8) for _ in range(depth)])
        self.norm=nn.LayerNorm(d,elementwise_affine=False,eps=1e-6); self.fmod=nn.Sequential(nn.SiLU(),nn.Linear(d,2*d)); self.out=nn.Linear(d,patch*patch)
        nn.init.normal_(self.pos,std=.02); nn.init.normal_(self.ye.weight,std=.02)
        nn.init.zeros_(self.fmod[-1].weight); nn.init.zeros_(self.fmod[-1].bias); nn.init.zeros_(self.out.weight); nn.init.zeros_(self.out.bias)
    def forward(self,x,t,h,y):
        b=x.shape[0]; x=self.pe(x).flatten(2).transpose(1,2)+self.pos; c=self.te(t)+self.he(h)+self.ye(y)
        for block in self.blocks: x=block(x,c)
        s,k=self.fmod(c).chunk(2,-1); x=self.norm(x)*(1+k[:,None,:])+s[:,None,:]; x=self.out(x)
        x=x.view(b,8,8,self.patch,self.patch,1); x=torch.einsum("nhwpqc->nchpwq",x)
        return x.reshape(b,1,32,32)

model=CondDiT().to(DEVICE)
print("params M:",sum(p.numel() for p in model.parameters())/1e6)


## 2. MeanFlow objective + Muon hybrid


In [ ]:
def logit_normal(n): return torch.sigmoid(torch.randn(n,device=DEVICE)*P_STD+P_MEAN)
def sample_tuple(images):
    a,b=logit_normal(len(images)),logit_normal(len(images)); t,r=torch.maximum(a,b),torch.minimum(a,b)
    r[:int(len(images)*DATA_PROPORTION)]=t[:int(len(images)*DATA_PROPORTION)]
    e=torch.randn_like(images); ti=t[:,None,None,None]; z=(1-ti)*images+ti*e
    return z,e-images,r,t

def mf_outputs(z,v,r,t,y):
    def fn(zv,tv,rv): return model(zv,tv,tv-rv,y)
    u,du=jvp(fn,(z,t,r),(v,torch.ones_like(t),torch.zeros_like(r)))
    target=(v-(t-r)[:,None,None,None]*du).detach(); return u,target

def loss_fn(u,target):
    sse=(u-target).pow(2).flatten(1).sum(1); w=(sse.detach()+NORM_EPS).reciprocal(); return (sse*w).mean(),sse.mean()

@torch.no_grad()
def ns5(g,steps=5,eps=1e-7):
    x=g.float(); tr=x.shape[0]>x.shape[1]
    if tr: x=x.T
    x=x/(x.norm()+eps); a,b,c=3.4445,-4.775,2.0315
    for _ in range(steps):
        A=x@x.T; x=a*x+(b*A+c*A@A)@x
    return (x.T if tr else x).to(g.dtype)

class MuonFallback(torch.optim.Optimizer):
    def __init__(self,params,lr=.02,momentum=.95): super().__init__(params,dict(lr=lr,momentum=momentum))
    @torch.no_grad()
    def step(self,closure=None):
        for group in self.param_groups:
            for p in group["params"]:
                if p.grad is None: continue
                state=self.state[p]; buf=state.setdefault("momentum_buffer",torch.zeros_like(p.grad)); buf.mul_(group["momentum"]).add_(p.grad)
                upd=ns5(p.grad+group["momentum"]*buf); upd*=math.sqrt(max(1.0,p.shape[0]/p.shape[1])); p.add_(upd,alpha=-group["lr"])

muon_params=[]; adam_params=[]
for name,p in model.named_parameters():
    if p.ndim==2 and (".attn." in name or ".mlp." in name): muon_params.append(p)
    else: adam_params.append(p)
if hasattr(torch.optim,"Muon"):
    opt_muon=torch.optim.Muon(muon_params,lr=MUON_LR,momentum=MUON_MOMENTUM,weight_decay=0.0); backend="torch.optim.Muon"
else:
    opt_muon=MuonFallback(muon_params,lr=MUON_LR,momentum=MUON_MOMENTUM); backend="fallback Muon"
opt_adam=torch.optim.AdamW(adam_params,lr=ADAMW_LR,betas=(.9,.99),eps=1e-8,weight_decay=0.0)
print("Muon backend:",backend,"Muon tensors:",len(muon_params),"AdamW tensors:",len(adam_params))


## 3. Conditional sampling / diagnostics / training


In [ ]:
fixed_noise=torch.randn(16,1,32,32,device=DEVICE); sample_rng=random.Random(SEED+10000)
diag_images,diag_labels=next(iter(test_loader)); diag_images=diag_images.to(DEVICE); diag_labels=diag_labels.to(DEVICE); diag_noise=torch.randn_like(diag_images)
da,db=logit_normal(len(diag_images)),logit_normal(len(diag_images)); diag_t,diag_r=torch.maximum(da,db),torch.minimum(da,db); diag_r[:int(len(diag_images)*DATA_PROPORTION)]=diag_t[:int(len(diag_images)*DATA_PROPORTION)]

@torch.no_grad()
def save_samples(step):
    model.eval(); ys=[sample_rng.choice(CONDITION_POOL) for _ in range(16)]; y=torch.tensor(ys,device=DEVICE)
    one=torch.ones(16,device=DEVICE); g=(fixed_noise-model(fixed_noise,one,one,y)).clamp(-1,1).add(1).div(2)
    fig,axs=plt.subplots(4,4,figsize=(7,7))
    for i,ax in enumerate(axs.flat): ax.imshow(g[i,0].cpu(),cmap="gray",vmin=0,vmax=1); ax.set_title(f"y={ys[i]}",fontsize=9); ax.axis("off")
    fig.suptitle(f"Conditional MeanFlow step {step}"); fig.tight_layout(); fig.savefig(os.path.join(SAMPLE_DIR,f"step_{step:05d}.png"),dpi=160,bbox_inches="tight"); plt.close(fig); model.train()

@torch.no_grad()
def diagnostics():
    model.eval(); ti=diag_t[:,None,None,None]; z=(1-ti)*diag_images+ti*diag_noise; v=diag_noise-diag_images; u,target=mf_outputs(z,v,diag_r,diag_t,diag_labels)
    mse=(u-target).pow(2).flatten(1).mean(1); cos=F.cosine_similarity(u.flatten(1),target.flatten(1),dim=1); inter=diag_r<diag_t; bound=diag_r==diag_t; model.train()
    return mse.mean().item(),mse[inter].mean().item(),cos[inter].mean().item(),mse[bound].mean().item()

def grad_norm(): return math.sqrt(sum(p.grad.detach().float().pow(2).sum().item() for p in model.parameters() if p.grad is not None))

def save_ckpt(step):
    torch.save({"step":step,"model":model.state_dict(),"muon":opt_muon.state_dict(),"adamw":opt_adam.state_dict(),"config":{"muon_lr":MUON_LR,"adamw_lr":ADAMW_LR,"norm_eps":NORM_EPS,"condition_pool":CONDITION_POOL}},os.path.join(CKPT_DIR,f"step_{step:05d}.pt"))

model.train(); it=iter(train_loader); start=time.time(); save_samples(0)
for step in range(1,TRAIN_STEPS+1):
    try: images,labels=next(it)
    except StopIteration: it=iter(train_loader); images,labels=next(it)
    images,labels=images.to(DEVICE,non_blocking=True),labels.to(DEVICE,non_blocking=True)
    z,v,r,t=sample_tuple(images); u,target=mf_outputs(z,v,r,t,labels); loss,raw=loss_fn(u,target)
    opt_muon.zero_grad(set_to_none=True); opt_adam.zero_grad(set_to_none=True); loss.backward(); gn=grad_norm(); opt_muon.step(); opt_adam.step()
    if step%LOG_EVERY==0:
        writer.add_scalar("train/loss_adaptive",loss.item(),step); writer.add_scalar("train/raw_sse",raw.item(),step); writer.add_scalar("train/grad_norm",gn,step); writer.add_scalar("run/elapsed_minutes",(time.time()-start)/60,step)
    if step%DIAG_EVERY==0:
        a,b,c,d=diagnostics(); writer.add_scalar("diagnostic/raw_mse_all",a,step); writer.add_scalar("diagnostic/raw_mse_interval",b,step); writer.add_scalar("diagnostic/interval_cosine",c,step); writer.add_scalar("diagnostic/boundary_mse",d,step); print(f"step={step:05d} loss={loss.item():.6f} grad={gn:.4f} mse={a:.4f} interval_cos={c:.4f} boundary={d:.4f}")
    if step%SAMPLE_EVERY==0: save_samples(step); save_ckpt(step)
    if step%250==0: writer.flush()
writer.flush(); torch.save({"step":TRAIN_STEPS,"model":model.state_dict()},os.path.join(RUN_DIR,"meanflow_cond_dit_muon_mnist_20k.pt")); writer.close()
print("saved:",RUN_DIR)
